In [ ]:
# the only package that this script requires is the random package
# otherwise, base python has everything we need
import random

# i did not use ChatGPT at all, and it's probably obvious

# here are objects which are globally defined

class Constants:
    """
    A container class for all constant values used in the card game.

    Attributes
    ----------
    SUITS : list of str
        The names of the four suits in the deck: Spades, Clubs, Diamonds, Hearts.

    ART : list of str
        Unicode symbols representing each suit visually: ♠︎, ♣︎, ♦︎, ♥︎.

    num_cards_per_suit : int
        Number of cards in each suit. For an Italian deck, this is set to 10.

    VALUES : range
        Numerical values assigned to the cards, from 1 up to `num_cards_per_suit`.

    SYMBOLS : list of str
        The printed symbols for each card value, such as 'A', '2', ..., '10'.
    """
    SUITS = ["Spades", "Clubs", "Diamonds", "Hearts"] # the suits
    ART=["♠︎","♣︎","♦︎","♥︎"]
    num_cards_per_suit = 10 # italian deck
    VALUES = range(1, num_cards_per_suit+1) # number of cards (can be changed)
    SYMBOLS = ['A','2','3','4','5','6','7','8','9','10']#,'J','Q','K'] # symbols associated with each card
    # POSITIONS = ["Hand","Tableau","Suitstacks"] # these are what I named each position of the board
    # the hand means the draw deck, the tableau means the table of cards in piles,
    # the suitstacks are where the suits are organized to win

    # here are where legal moves for the game are defined
    # it takes ``cartota`` ``playota`` and ``board`` as input
    # sorry for the weird and annoying names of the variables
    # the first two objects, ``cartota and ``playota`` are tuples
    # cartota = (Card, (position, extra position info)) and same formatting for ``playota``
    # the cartota is the card which is being played
    # and the playota is the card where playing on
    # the Card class is defined below
    # the position is a string, ^see above line POSITIONS
    # the extra info means where in that position our card is
    # for the Hand it is always None, for the Tableau, it says which pile we are in
    # for the Suitstacks it says which suit we are in
    # this function is called in the ``check_playable()`` method of the Board class (defined below)

    
    def legal_move(cartota, playota, board): # returns whether cartota can be played on playota, True or False

        """
    Determine whether a given card can be legally played onto another card/location.

    Parameters
    ----------
    cartota : tuple
        A tuple describing the card being played.
        Format: (card, (location, extra_info))
        - card: the Card object being moved.
        - location: "Tableau" or "Hand".
        - extra_info: index or additional info depending on location.

    playota : tuple
        A tuple describing the card or slot being played onto.
        Format: (playon_card, (location, extra_info))
        - playon_card: the Card object currently present, or an empty placeholder.
        - location: "Tableau" or "Suitstacks".
        - extra_info: index or suit depending on location.

    board : Board
        The current game board object. Needed to check Tableau pile structure
        and whether a card being moved from the Tableau is unblocked.

    Returns
    -------
    bool
        True if the move is legal according to game rules, False otherwise.

    Notes
    -----
    Rules enforced:
    - Tableau:
        * If the destination spot is empty, only the highest-valued card
          (e.g., King in a standard deck) can be placed.
        * Otherwise, the move is legal only if the two cards are "braided"
          (opposite color + descending value).
    - Suitstacks:
        * Card must match the suit of the target stack.
        * If the stack is empty, only the lowest-valued card (e.g., Ace) can be placed.
        * Otherwise, cards must be stacked in ascending order.
        * If moving from the Tableau, the card must not be blocked by another face-up card.
    """


        card = cartota[0]
        location = cartota[1][0]
        loc_extra = cartota[1][1]

        playon_card = playota[0]
        playon_loc = playota[1][0]
        playon_extra = playota[1][1]


        if playon_loc == "Tableau": # if we are playing on the Tableau
            if playon_card.empty: # if that spot in the Tableau is empty
                return card.value == Constants().VALUES[-1] # only the King can go there
            
            # otherwise tell us if the cards are "braided" (see below)
            # meaning opposite color and descending value
            else: return card.is_braided_with(playon_card)
            
        else: # playon_loc == "Suitstacks": # we are playing on the suitstacks
            if playon_extra == card.suit: # the card must go in the appropriate suitstack, i.e. matching suits
                if playon_card.empty: return card.value == Constants().VALUES[0] # if the stack is empty, only the Ace goes there

                # if we are playing from the Tableau, double check that we are using an unblocked card
                # otherwise, the location is the hand
                elif location == "Tableau" and board.tableau.piles[loc_extra].face_up[-1] == card or location == "Hand":
                    return card.value - playon_card.value == 1 # the cards in the suitstacks must be in ascending order
                
                else: return False

            else: return False
   
           

# this is the Card class
# a non-empty card has a value and a suit
# otherwise, it is empty.  it is useful to define empty spaces for the solitaire rules^
class Card:
    """
    Represents a single playing card or an empty placeholder card.

    Parameters
    ----------
    value : int, optional
        Numerical value of the card (e.g., 1–10 for an Italian deck).
        Ignored if `empty=True`.

    suit : str, optional
        The card's suit: must be one of the suits defined in `Constants.SUITS`.
        Ignored if `empty=True`.

    empty : bool, default False
        If True, creates an empty placeholder card used to represent an empty slot
        in the Tableau or Suitstacks.

    Attributes
    ----------
    empty : bool
        True if this is an empty placeholder card.

    name : str
        card name such as "A of Spades" or "10 of Hearts".
        For empty cards, this is "Empty".

    suit : str
        Card suit ("Spades", "Clubs", "Diamonds", "Hearts"). Not present if empty.

    color : str
        Card color ("Black" for Spades/Clubs, "Red" for Diamonds/Hearts).
        Not present if empty.

    value : int
        Numeric value of the card. Not present if empty.

    symbol : str
        Short printable representation, e.g. "A♠︎", "7♥︎".
        Not present if empty.

    Notes
    -----
    - Empty cards are used instead of `None` to simplify game logic.
    - A card's color is derived automatically from its suit.
    """
    
    def __init__(self, value=None, suit=None, empty=False):
        if empty==True:
            self.empty=True
            self.name = "Empty"
        else:
            self.suit = suit
            if self.suit == "Spades" or self.suit == "Clubs": self.color = "Black"
            else: self.color = "Red"
            self.value = value
            self.name = str(Constants().SYMBOLS[self.value-1])+" of "+self.suit # this makes it easy to identify cards in a print
            self.symbol=str(Constants().SYMBOLS[self.value-1])+Constants().ART[Constants().SUITS.index(self.suit)]
            self.empty=False

            ##################################################3

    
    # a "braid" is when two cards are separated by 1 in value and opposite colors
    def is_braided_with(self, playon_card): return playon_card.value - self.value == 1 and playon_card.color != self.color
    """
    Determine whether this card can be placed onto another card
    according to the "braided" rule.

    Parameters
    ----------
    playon_card : Card
        The card already present in the Tableau onto which `self` is being played.

    Returns
    -------
    bool
        True if:
        - `self` has a value exactly one lower than `playon_card`, and
        - The two cards have opposite colors (Red vs Black).
        
        False otherwise.

    Notes
    -----
    This rule corresponds to the standard descending-opposite-color constraint
    seen in many solitaire-style games.
    """



# the deck class essentially tells us how to shuffle and deal
# if we want to specify some deck order, then we can do so.
class Deck:
    """
    Represents a standard deck of cards for the game.

    The Deck can be initialized either in a default ordered state
    or with a custom order of cards.

    Parameters
    ----------
    desired_order : list of Card, optional
        If provided, this list of Card objects is used to initialize
        the deck in the desired order. If not provided, the deck is
        automatically filled with cards in standard order
        (all suits in order with values from Constants.VALUES).

    Attributes
    ----------
    cards : list of Card
        List of Card objects currently in the deck, in order.

    Notes
    -----
    - The default order is created by iterating through Constants.SUITS
      and Constants.VALUES in order, creating a Card for each combination.
    - This deck can be used for shuffling, drawing, or initializing
      the game board.
    """
    def __init__(self, desired_order=None):
        self.cards = []

        if desired_order is not None: self.cards = desired_order

        else: # just put the cards in order
            for suit in Constants().SUITS:
                for value in Constants().VALUES:
                    self.cards += [Card(value,suit)]

                    
    def shuffle(self): # then shuffle them
        """
    Randomly shuffle the deck of cards in place.

    This method uses Python's built-in `random.shuffle` function to
    reorder the `self.cards` list randomly.

    Notes
    -----
    - Shuffling modifies the deck in place and does not return a new deck.
    - This method should be called before dealing or drawing cards
      to ensure random card order.
     """
        random.shuffle(self.cards) # the ``shuffle`` method of ``random`` does exactly that
         
    # here is the particular way in which cards are dealt
    def deal(self):
        """
    Deal the cards into a Solitaire game layout.

    This method distributes the deck into:
    - Tableau: 6 piles with face-up and face-down cards in a triangular pattern.
    - Hand: the remaining cards after filling the Tableau.
    - Suitstacks: empty stacks for each suit, ready to receive cards during gameplay.

    Returns
    -------
    Board
        A Board object containing:
        - `tableau`: initialized with Pile objects, each containing face-up and face-down cards.
        - `hand`: a Hand object containing the remaining cards.
        - `suitstacks`: a Suitstacks object with empty stacks for each suit.

    Dealing Logic
    -------------
    - Number of piles in the Tableau: 6
    - Total cards in the Tableau: 21 (using formula n/2*(n+1) for triangular arrangement)
    - For the first pile, all cards are face-up.
    - For other piles, only the last card is face-up, the rest are face-down.
    - Remaining cards go into the Hand.
    - Suitstacks are initialized empty for each suit.

    Notes
    -----
    - Relies on the following classes being defined:
        * `Pile(face_up_cards, face_down_cards, pile_index)`
        * `Tableau(piles)`
        * `Hand(cards)`
        * `Stack(suit, cards=None)`
        * `Suitstacks(stacks)`
        * `Board(tableau, hand, suitstacks)`
    - This method does not modify the original deck order beyond slicing.
    """
        num_piles = 6 # 6 piles
        num_tableau = int(num_piles/2*(num_piles+1)) # 21 total in the tableau
        piles=[]
        for i in range(num_piles): # this loop creates the piles
            cards=self.cards[int(i/2*(i+1)):int((i+1)/2*(i+2))]
            if i==0: face_up,face_down = (cards,[]) # the first pile has no face down cards
            else: face_up,face_down=(cards[-1:],cards[:-1]) # otherwise only the last card is face up

            # the Pile class is defined below, it is initialized with face up cards, face down cards, and an index
            # which indicates which pile it is on the tableau
            piles += [Pile(face_up_cards=face_up, face_down_cards=face_down, pile_index=i)]

        # the Tableau class is defined below, it is initialized with a list of piles
        tableau = Tableau(piles=piles)

        # the Hand class is defined below, it is initialized with a list of cards
        hand = Hand(self.cards[num_tableau:]) # the rest of the cards not in the Tableau are put into the Hand

        # the Suitstacks class is defined below as well as the Stack class
        # Suitstacks are initialized with stacks and a Stack is initialized with a Suit and a potential list of cards
        # here, when the cards are dealt, all stacks are empty, but we can initialize a board with nonempty stacks
        suitstacks = Suitstacks(stacks=[Stack(suit=suite) for suite in Constants().SUITS])

        # the Board class is defined below, it is initialized with a Tableau, a Hand, and a Suitstacks
        # this is meant to represent a game of Solitaire
        return Board(tableau=tableau, hand=hand, suitstacks=suitstacks)


    
# each object: Tableau, Hand, and Suitstacks will have a method called ``available()``
# these will be called in the Board method ``check_available()`` telling us which cards are available to play
# ``available()`` returns a list of ``cartota`` objects (see above for cartota definition)
# out of all of the available cards, we then see if there exist any legal moves for that card using ``legal_move`` method of Constants

# Hand is initialized with a list of Card objects
class Hand:
    """
    Represents the player's hand in a Solitaire-style card game.

    The hand contains cards that are not currently on the Tableau or Suitstacks,
    and all cards in the hand are considered available for play.

    Attributes
    ----------
    cards : list of Card
        The list of Card objects currently in the hand.
    """
    def __init__(self, cards_in_hand):
        """
        Initialize the Hand with a list of cards.

        Parameters
        ----------
        cards_in_hand : list of Card
            The cards to include in the player's hand.
        """
        self.cards = cards_in_hand
    
    # all cards in the hand are available to play
    # unless the hand has no cards
    def available(self):
        """
        Return all cards in the hand as available moves.

        Returns
        -------
        list of tuple
            A list of tuples representing available moves from the hand.
            Each tuple is formatted as (card, ("Hand", None)).

            - card: Card object in the hand.
            - ("Hand", None): indicates the location is the hand and no extra info is needed.

        Notes
        -----
        - If the hand has no cards, returns an empty list.
        """
        if len(self.cards) == 0: return []
        else: return [(card,("Hand",None)) for card in self.cards]
    
    # how to copy
    def copy(self): return Hand(cards_in_hand=self.cards.copy())
    """
        Create and return a copy of the hand.

        Returns
        -------
        Hand
            A new Hand object containing a shallow copy of the cards list.

        Notes
        -----
        - This method is useful for simulating moves or branching game states
          without modifying the original hand.
        """
        
# Tableau is initialized with a list of Pile objects
class Tableau:
    """
    Represents the Tableau in a Solitaire-style card game.

    The Tableau consists of multiple piles of cards. Some cards are face-up
    (available for play) and others are face-down (hidden). This class manages
    access to available cards and allows copying the Tableau state.

    Attributes
    ----------
    piles : list of Pile
        The list of Pile objects that make up the Tableau.
    """
    def __init__(self, piles):
        """
        Initialize the Tableau with a list of piles.

        Parameters
        ----------
        piles : list of Pile
            The Pile objects that form the Tableau.
        """
        self.piles = piles
    
    # all face up cards in the Tableau are available
    def available(self):
        """
        Return all face-up cards in the Tableau that are available to play.

        Returns
        -------
        list of tuple
            A list of tuples representing available cards in the Tableau.
            Each tuple is formatted as (card, ("Tableau", pile_index)).

            - card: Card object that is available.
            - ("Tableau", pile_index): indicates the card's location in the Tableau
              and which pile it belongs to.

        Notes
        -----
        - Cards are considered available if they are face-up and not the last
          highest-value card in an empty pile (e.g., King with no face-down cards).
        - Skips empty piles.
        """
        available = []
        for pile in self.piles:
            if pile.is_empty(): continue
            
            for card in pile.face_up:
                if card.value == Constants().VALUES[-1] and len(pile.face_down) == 0: continue
                available += [(card,("Tableau",pile.pile_index))]
        return available

    # copy each pile, see below
    def copy(self): return Tableau(piles=[pile.copy() for pile in self.piles])
    """
        Create and return a copy of the Tableau.

        Returns
        -------
        Tableau
            A new Tableau object containing copies of all piles.

        Notes
        -----
        - Each Pile is assumed to have its own `.copy()` method that returns an
          independent copy of the pile, including its face-up and face-down cards.
        - Useful for simulating moves or maintaining independent game states.
        """
        


# Pile is initialized with a list of face down cards, a list of face up cards, and a pile index
# the card lists are lists of Card objects, and the pile_index is an integer from 0 to 6
class Pile:
    """
    Represents a single pile of cards in the Tableau of a Solitaire-style game.

    Each pile consists of face-down cards and face-up cards. The class provides
    utility methods to check if the pile is empty, retrieve all cards (with an
    initial empty placeholder), and create a copy of the pile.

    Attributes
    ----------
    pile_index : int
        Index of the pile within the Tableau.

    face_up : list of Card
        List of face-up Card objects in the pile.

    face_down : list of Card
        List of face-down Card objects in the pile.
    """
    def __init__(self, face_down_cards, face_up_cards, pile_index):
        """
        Initialize a Pile with face-down and face-up cards and its index.

        Parameters
        ----------
        face_down_cards : list of Card
            Cards that are currently face-down in the pile.

        face_up_cards : list of Card
            Cards that are currently face-up in the pile.

        pile_index : int
            The index of the pile in the Tableau.
        """
        self.pile_index = pile_index
        self.face_up = face_up_cards
        self.face_down = face_down_cards
    
    # it's useful to know if a pile is empty
    def is_empty(self):
        """
        Check whether the pile is effectively empty.

        Returns
        -------
        bool
            True if the pile contains only the initial empty Card placeholder,
            otherwise False.

        Notes
        -----
        - Uses the `cards()` method which always includes an empty Card at the front.
        """
        # here ``cards`` is a method rather than an attribute
        if len(self.cards())==1: return True
    
    # i decided to do a method because we update the face up and face down cards
    # and we always want an ``empty`` Card to be at the front of the list
    # sometimes self.face_up = [] will be an empty list and likewise self.face_down
    def cards(self): return [Card(empty=True)] + self.face_down + self.face_up

    """
        Return a list of all cards in the pile, including an initial empty placeholder.

        Returns
        -------
        list of Card
            The pile as a list of cards in the order:
            [empty placeholder, face-down cards, face-up cards].

        Notes
        -----
        - The empty Card at the front simplifies game logic by ensuring the pile
          is never truly "empty" from a list perspective.
        - Updates dynamically with changes to `face_up` and `face_down`.
        """

    # copy the pile
    def copy(self): return Pile(face_down_cards=self.face_down.copy(),face_up_cards=self.face_up.copy(),pile_index=self.pile_index)

    """
        Create and return a copy of the pile.

        Returns
        -------
        Pile
            A new Pile object with independent copies of the face-down and
            face-up cards and the same pile index.

        Notes
        -----
        - Useful for simulating moves or maintaining independent game states
          without affecting the original pile.
        """


# Suitstacks is initialized with a list of Stack objects
class Suitstacks:
    """
    Represents the suit-specific stacks in a Solitaire-style card game.

    Each Suitstack corresponds to one suit (Spades, Clubs, Diamonds, Hearts)
    and holds cards in ascending order. Only the top card of a non-empty
    stack is considered available for play.

    Attributes
    ----------
    stacks : dict
        Dictionary mapping suit names (strings) to Stack objects.
        Example: {'Spades': Stack, 'Hearts': Stack, ...}
    """
    def __init__(self, stacks):
        """
        Initialize Suitstacks with a list of Stack objects.

        Parameters
        ----------
        stacks : list of Stack
            List of Stack objects, one for each suit.

        Notes
        -----
        - The dictionary `self.stacks` uses the suit name as the key
          and the Stack object as the value.
        """
        # this is a dictionary where the key is one of the strings in Constants().SUITS
        self.stacks = {stack.suit:stack for stack in stacks}

    # if the suitstack is not empty, then its available cards is only the one on top
    def available(self):
        """
        Return all cards in the Suitstacks that are currently available to play.

        Returns
        -------
        list of tuple
            A list of tuples representing available cards.
            Each tuple is formatted as (card, ("Suitstacks", suit)).

            - card: the top Card object of the stack (if stack is not empty)
            - ("Suitstacks", suit): indicates the location and suit of the card

        Notes
        -----
        - Only the top card of each non-empty stack is available.
        - Empty stacks are skipped.
        """
        available = []
        for suit in Constants().SUITS:
            stack = self.stacks[suit]
            if stack.is_empty(): continue
            available += [(stack.cards[-1],("Suitstacks",suit))]
        return available
    
    # copy each stack, see below
    def copy(self): return Suitstacks(stacks=[self.stacks[suit].copy() for suit in Constants().SUITS])
    """
        Create and return a copy of all Suitstacks.

        Returns
        -------
        Suitstacks
            A new Suitstacks object containing copies of all individual stacks.

        Notes
        -----
        - Each Stack is assumed to have its own `.copy()` method that returns
          an independent copy of the stack.
        - Useful for simulating moves or maintaining independent game states.
        """


# a stack object is initialized with a suit (string from Constants().SUITS) and a list of Card objects
# by default, it is empty
class Stack:
    """
    Represents a single suit stack in a Solitaire-style game.

    Each Stack corresponds to a single suit (Spades, Clubs, Diamonds, Hearts)
    and holds cards in ascending order. The stack always contains an initial
    empty placeholder card to simplify game logic.

    Attributes
    ----------
    suit : str
        The suit of the stack (e.g., "Spades", "Hearts", etc.).

    cards : list of Card
        List of Card objects in the stack, including an initial empty Card.
    """
    
    def __init__(self, suit, cards=[]):
        """
        Initialize a Stack with a suit and optional list of cards.

        Parameters
        ----------
        suit : str
            The suit for this stack.

        cards : list of Card, optional
            List of Card objects to include in the stack. Defaults to an empty list.

        Notes
        -----
        - An empty Card placeholder is automatically added at the front of the stack
          to simplify game logic.
        """
        
        self.suit = suit
        self.cards = [Card(empty=True)] + cards

    # it's useful to know when the suitstack is empty
    def is_empty(self):
        if len(self.cards)==1: return True
        """
        Check whether the stack is effectively empty.

        Returns
        -------
        bool
            True if the stack contains only the initial empty Card placeholder,
            otherwise False.

        Notes
        -----
        - This method is used to determine if the Suitstack has any playable cards.
        """

    # copy the stack (remember to exclude the empty card)
    def copy(self): return Stack(suit=self.suit,cards=self.cards.copy()[1:])
    """
        Create and return a copy of the stack (excluding the empty placeholder).

        Returns
        -------
        Stack
            A new Stack object with the same suit and an independent copy of
            the card list (excluding the initial empty card).

        Notes
        -----
        - Useful for simulating moves or maintaining independent game states.
        """
        
    
# Board is initialized with a Hand object, a Tableau object, and a Suitstacks object
# each of these will correspond to attributes of the Board
class Board:
    """
    Represents the overall state of a Solitaire-style card game.

    The Board contains all components of the game:
    - The player's hand
    - The Tableau (piles of face-up and face-down cards)
    - Suitstacks (one for each suit)
    - A flag indicating if no moves are available

    Attributes
    ----------
    hand : Hand
        The player's hand, containing cards not on the Tableau or Suitstacks.

    tableau : Tableau
        The Tableau object containing all piles of cards.

    suitstacks : Suitstacks
        The Suitstacks object containing stacks for each suit.

    no_moves : bool
        Flag to indicate whether there are no legal moves available.
        Defaults to False.

    store_history : bool
        Placeholder for a feature to store move history (not implemented).
        Defaults to False.
    """
    def __init__(self, hand, tableau, suitstacks, store_history = False): # i have not implemented the history
        """
        Initialize a Board with the given hand, Tableau, and Suitstacks.

        Parameters
        ----------
        hand : Hand
            The player's hand containing available cards.

        tableau : Tableau
            The Tableau containing piles of face-up and face-down cards.

        suitstacks : Suitstacks
            The suit-specific stacks for building sequences in ascending order.

        store_history : bool, optional
            Placeholder to track game history. Defaults to False.

        Notes
        -----
        - `no_moves` is initialized to False and can be updated during gameplay
          to indicate when no legal moves remain.
        """
        
        self.hand = hand
        self.tableau = tableau
        self.suitstacks = suitstacks
        self.no_moves = False

        
    # the script will evolve the solitaire game randomly
    def random_play(self):
        """
    Perform a random legal move on the board, if any are available.

    The method selects a random card from the currently available cards,
    checks all possible legal moves for that card, and executes one randomly
    if there are legal options. If no moves are available for a selected card,
    it is removed from consideration until all options are exhausted.

    Steps
    -----
    1. Retrieve all available cards using `check_available()`.
    2. Randomly select one card (`cartota`) from the available list.
    3. Check which locations the selected card can be legally played on using `check_playable()`.
    4. If legal moves exist:
       - Select one at random.
       - Execute the move using the `move()` method.
       - Break the loop after a successful move.
    5. If no legal moves exist for the selected card, remove it from the available list and repeat.

    Notes
    -----
    - A "cartota" is a tuple representing a card and its current location.
    - This method modifies the board in place.
    - Random selection introduces non-deterministic behavior and can be used
      for simulation, testing, or AI baseline strategies.
    """
        available = self.check_available() # check what's available (see below)
        while len(available)>0: # we filter cards which have no legal moves
            cartota = random.choice(available) # select a random available card (a cartota is a tuple, see above)
            playable = self.check_playable(cartota) # check what we can legally play on (see below)
            
            # if we can play
            if len(playable) != 0:
                # then choose a random move (see below for ``move`` method definition
                self.move(cartota_move=cartota, playota_onto=random.choice(playable))
                break # break the loop once we have moved

            # if there are no legal moves for this card, then remove it from the list
            else: available.remove(cartota)

    
    # this form of play will take a strategy as input
    # ``strategy`` is a tuple of functions: a heuristic and a priority function
    # the game will evolve based on which move has the highest heuristic value
    def greed_play(self, strategy):
        """
    Perform a move on the board using a greedy strategy with heuristic evaluation and tie-breaking priorities.

    Parameters
    ----------
    strategy : tuple
        A tuple containing two functions: (heuristic, priority)
        - heuristic(cartota_move, playota_onto, board) -> numeric score
            Evaluates the quality of a potential move.
        - priority(cartota_move, playota_onto, board) -> numeric score
            Used to break ties when multiple moves have the same highest heuristic.

    Behavior
    --------
    1. Retrieves all available cards on the board using `check_available()`.
    2. For each available card, identifies all legal destinations (`check_playable()`).
    3. Computes heuristic scores for each potential move.
    4. Selects moves with the maximum heuristic value:
        - If only one move has the best heuristic, execute it.
        - If multiple moves share the highest heuristic, compute priority scores to break ties.
        - If there is still a tie after priorities, choose randomly among the best moves.
    5. Updates the board in place by executing the chosen move.
    6. If no legal moves are available, sets `self.no_moves = True`.

    Notes
    -----
    - A "cartota" is a tuple representing a card and its current location.
    - A "playota" is a tuple representing a potential destination for a card.
    - This method combines heuristic evaluation and tie-breaking priorities to implement
      a greedy AI strategy for automated gameplay.
    - Random selection among tied moves introduces non-deterministic behavior.
    """
        

        available = self.check_available()

        # this will be a list which contains all plays for every card in available
        # often the elements will be empty
        able_moves=[]

        # here is the heuristic and priority
        heuristic=strategy[0]
        priority=strategy[1]
        
        # h will be the heuristic for each play
        # so it will have the same dimension as all_plays
        h=[]
        # we will save which locations in ``h`` have the highest values
        outer_indices=[]
        inner_indices=[]
        best_scores=[] # this is a list of best heuristic scores for each element of ``available``

        for able in available:
            playable = self.check_playable(able)
            able_moves+=[playable]

            scores=[]
            if len(playable)==0: scores+=[-100] # make sure that we never have to look at cards which have no moves
            else: scores+=[heuristic(cartota_move=able,playota_onto=play,board=self) for play in playable] # evaluate heuristic
            h+=[scores] # save these scores in a list

            best_score=max(scores)
            best_scores+=[best_score]

            # these will tell us which plays have the highest heuristic
            # for a given card on the board
            # much of the time this value will just be -100
            inds=[]
            for i in range(len(scores)):
                if scores[i] == best_score: inds+=[i]
            outer_indices+=[inds]

        # this tells us which cards have the best heuristic
        best=max(best_scores)
        inner_indices=[]
        for j in range(len(best_scores)):
            if best_scores[j]==best: inner_indices+=[j]
        
        # multiple moves may have the highest heuristic value
        # this is often the case
        best_moves=[]
        for i in inner_indices:
            if able_moves[i]==[]: continue
            for j in outer_indices[i]:
                best_moves += [(available[i],able_moves[i][j])]
        
        if len(best_moves)==0: self.no_moves = True

        # if there is only one best move we can go ahead and play it
        elif len(best_moves)==1: self.move(cartota_move=best_moves[0][0],playota_onto=best_moves[0][1])

        # otherwise we break the tie using priorities
        else:
            p=[] # list of priorities
            p+=[priority(cartota_move=best_moves[i][0], playota_onto=best_moves[i][1], board=self) for i in range(len(best_moves))]

            best_p=max(p) # find the highest priority
            prio_moves=[]
            for i in range(len(p)):
                if p[i]==best_p: prio_moves+=[best_moves[i]]
            
            # if we have a tie among priorities, then choose randomly
            if len(prio_moves)==1: self.move(cartota_move=prio_moves[0][0],playota_onto=prio_moves[0][1])
            else:
                chosen_move = random.choice(prio_moves)
                self.move(cartota_move=chosen_move[0],playota_onto=chosen_move[1])


    def check_available(self): # look at all parts of the board and see what's available
        """
    Retrieve all currently available cards on the board that can potentially be played.

    Returns
    -------
    list of tuple
        A list of "cartota" tuples representing available cards and their locations.
        Each tuple is formatted as (card, (location_type, location_extra)):

        - card : Card object that is available to play.
        - location_type : str, one of {"Hand", "Tableau", "Suitstacks"}.
        - location_extra : additional info depending on the location:
            * If Tableau: pile index (int)
            * If Suitstacks: suit name (str)
            * If Hand: None

    Behavior
    --------
    - Combines available cards from all components of the board:
        * Hand
        * Tableau (all face-up cards in each pile, following game rules)
        * Suitstacks (top card of each non-empty stack)
    - Returns a single consolidated list of all available cards.

    Notes
    -----
    - Does not check whether a move is legal; only identifies cards that are
      in a position to be moved according to board rules.
    - Useful for move generation, heuristics, and AI strategies.
    """
        hand_available = self.hand.available()
        tableau_available = self.tableau.available()
        suitstacks_available = self.suitstacks.available()
        return hand_available + tableau_available + suitstacks_available # a big list of cartota tuples


    # for this method we pass a cartota, which is what i call a tuple of a Card object and its place on the board (see above)
    def check_playable(self, cartota):
        # look the last card in each pile (which includes empty spaces)
        # and look at the last card in each suistack (also includes empty spaces)
        # these are in general the kind of places we can play
        """
    Determine all legal target locations where a given card can be played.

    Parameters
    ----------
    cartota : tuple
        A tuple representing the card to move and its current location.
        Format: (card, (location_type, location_extra))
        - card : Card object to be played
        - location_type : str, one of {"Hand", "Tableau", "Suitstacks"}
        - location_extra : additional info depending on location:
            * If Tableau: pile index (int)
            * If Suitstacks: suit name (str)
            * If Hand: None

    Returns
    -------
    list of tuple
        A list of tuples representing valid play targets ("playotas") for the card.
        Each tuple is formatted as (target_card, (target_location, extra_info)).

        - target_card: the top Card object in the potential target pile or stack.
        - (target_location, extra_info): indicates where the card can be legally played.
          * For Tableau: pile index (int)
          * For Suitstacks: suit name (str)

    Behavior
    --------
    - Checks the top card of each Tableau pile and each Suitstack (including empty spaces).
    - Filters out any locations where the move would be illegal according to
      `Constants.legal_move(cartota, playota, board)`.
    - Returns only the playable targets for the given card.

    Notes
    -----
    - "cartota" refers to the card being moved along with its current location.
    - "playota" refers to a potential destination for the card.
    - This method does not modify the board state; it only identifies legal moves.
    """
        all_plays = []
        for pile in self.tableau.piles:
            all_plays += [(pile.cards()[-1], ("Tableau",pile.pile_index))]
        for suit in Constants().SUITS:
            stack = self.suitstacks.stacks[suit]
            all_plays += [(stack.cards[-1], ("Suitstacks",suit))]
        
        # a playota is the same as a cartota, a tuple of a Card object and its place on the board
        # i just call it playota so we know which card we are playing and which one is being played on
        dum=all_plays.copy()
        for play in all_plays:
            # return only playotas which the cartota can legally move to
            if not Constants.legal_move(cartota=cartota, playota=play, board=self): dum.remove(play)
        
        return dum
        
        
    # this is how cards move on boards
    # ``cartota_move`` is a cartota of the card we are moving
    # ``playota_onto`` is a cartota we are playing on (playota)
    # we already assume that the cards have been filtered in the ``check_playable`` method
    def move(self, cartota_move, playota_onto):
        """
    Execute a move on the board by transferring a card from its current location
    to a target location (Tableau or Suitstacks).

    Parameters
    ----------
    cartota_move : tuple
        A tuple representing the card to move and its current location.
        Format: (card, (location_type, location_extra))
        - card : Card object to be moved
        - location_type : str, one of {"Hand", "Tableau", "Suitstacks"}
        - location_extra : additional info depending on location:
            * If Tableau: pile index (int)
            * If Suitstacks: suit name (str)
            * If Hand: None

    playota_onto : tuple
        A tuple representing the destination where the card will be played.
        Format: (card_or_none, (destination_type, destination_extra))
        - destination_type : str, one of {"Tableau", "Suitstacks"}
        - destination_extra : additional info depending on destination:
            * If Tableau: pile index (int)
            * If Suitstacks: suit name (str)

    Behavior
    --------
    - If moving from the Hand:
        * Remove the card from the hand.
        * Add the card to the target Tableau pile or Suitstack.

    - If moving from the Tableau:
        * Move the selected card along with any cards on top of it (a "chunk").
        * If the moved card was blocking a face-down card, flip the top face-down card.
        * Place the chunk on the target Tableau pile or add the card to a Suitstack.

    - If moving from a Suitstack:
        * Remove the card from the Suitstack.
        * Add the card to the target Tableau pile (only legal destination).

    Notes
    -----
    - Assumes legality of the move has already been checked.
    - Modifies the board in place, updating Hand, Tableau, and Suitstacks accordingly.
    - Handles multi-card moves from Tableau piles, flipping face-down cards when needed.
    """
        card = cartota_move[0]
        location = cartota_move[1][0]
        loc_extra = cartota_move[1][1]

        # play = playota_onto[0] # turns out we don't need to know this
        play_loc = playota_onto[1][0]
        play_extra = playota_onto[1][1]

        if location == "Hand": # if we are playing from the hand
            self.hand.cards.remove(card) # remove the card from the hand

            if play_loc == "Tableau": # if we are playing onto the Tableau

                # then add that card to the front of the Tableau
                self.tableau.piles[play_extra].face_up += [card]

            else: # play_loc == "Suitstacks": # otherwise we must be playing on the suitstacks
                self.suitstacks.stacks[play_extra].cards += [card] # add the card to the top of the suitstack
        
        elif location == "Tableau": # otherwise if we are playing from the tableau

            # when a blocked card is moved from the tableau, we have to move all of the cards below it
            moving_pile = self.tableau.piles[loc_extra].face_up # these are all face up cards in the pile of interest
            position = moving_pile.index(card) # this tells us where our card of interest lies in the pile
            chunk = moving_pile[position:] # we move the card along with everything that follows
            self.tableau.piles[loc_extra].face_up = moving_pile[:position] # we remove all of these cards from the face up cards of that pile

            # sometimes we are moving an unblocked card from the tableau which is blocking a face down card
            if position == 0 and len(self.tableau.piles[loc_extra].face_down) > 0:
                flip = self.tableau.piles[loc_extra].face_down[-1] # this is the card we flip from face down to face up
                self.tableau.piles[loc_extra].face_down.remove(flip) # it is no longer face down
                self.tableau.piles[loc_extra].face_up = [flip] # it is now face up

            if play_loc == "Tableau": # if we are playing onto another card in the tableau
                self.tableau.piles[play_extra].face_up += chunk # then slap the chunk on that card
            
            else: # play_loc == "Suitstacks": # otherwise we must playing onto the suitstacks
                self.suitstacks.stacks[play_extra].cards += [card] # so we add that card to the stack
                # note that we have filtered inappropriate plays from tableau onto the suistacks
            
        else: # location == "Suistacks": # if we are not playing from the Hand or Tableau, then we must be playing from the suistacks
            self.suitstacks.stacks[loc_extra].cards.remove(card) # remove that card from the stack
            self.tableau.piles[play_extra].face_up += [card] # add that card to the pile (we can only play from the suitstacks to a tableau pile)

    
    def show(self): # print out which cards in what parts of the board
        """
    Display a readable summary of the current board state.

    This method prints:
    - The number of cards currently in the player's hand.
    - Each Tableau pile, showing:
        * face-down cards (listed by name)
        * face-up cards (listed by name)
    - The Suitstacks, showing how many cards have been placed in each suit.

    The output is formatted for quick human inspection during gameplay or
    debugging, without using any graphical interface.

    Notes
    -----
    - Assumes:
        * `self.hand.cards` is a list of Card objects.
        * `self.tableau.piles` is a list of pile objects, each having
          `face_down` and `face_up` lists.
        * `self.suitstacks.stacks[suit].cards` contains an initial empty card
          used as a placeholder; this method subtracts 1 to report the true
          number of placed cards.
    - This method prints directly to the console and returns nothing.
    """

        # i wanted to implement some cool text art but did not have time
        print("no. cards in hand: "+str(len(self.hand.cards))+"\n")
        for pile in self.tableau.piles:
            st="pile: "
            st+="\n\tface down: "
            for card in pile.face_down: st+=card.name+","
            st+="\n\tface up: "
            for card in pile.face_up: st+=card.name+","
            print(st)
        st2="\nSuitstacks:\n\t"
        for suit in Constants().SUITS:
            st2+=suit+": "+str(len(self.suitstacks.stacks[suit].cards)-1)+"\t"
        print(st2)
   

    # how to copy the board
    def copy(self): return Board(hand=self.hand.copy(),tableau=self.tableau.copy(),suitstacks=self.suitstacks.copy())
    """
    Create and return a deep copy of the current Board object.

    Returns
    -------
    Board
        A new Board instance containing independent copies of:
        - hand
        - tableau
        - suitstacks

    Purpose
    -------
    This method is used to safely simulate moves without altering the
    original game state. Useful for:
    - AI search algorithms
    - heuristic evaluation
    - undo/redo functionality
    - branching possible move sequences

    Notes
    -----
    - Assumes that `hand`, `tableau`, and `suitstacks` each implement
      their own `.copy()` method that returns a structurally independent
      copy.
    - This is not a Python `copy.deepcopy()`; rather, it creates a new
      Board using the object's internal copying methods.
    """

    def check_win(self): return all([len(pile.face_down)==0 for pile in self.tableau.piles])
    """
    Check whether the game has been won.

    Returns
    -------
    bool
        True if *all* tableau piles have zero face-down cards remaining,
        meaning the player has successfully uncovered every card.
        False otherwise.

    Explanation
    -----------
    In many solitaire-style games, the win condition is satisfied when
    every face-down card in the Tableau has been flipped. This method
    checks each tableau pile and returns True only if all piles contain
    no face-down cards.

    Notes
    -----
    - This method assumes `self.tableau.piles` is a list of pile objects,
      each containing a list attribute `face_down`.
    - This does *not* verify Suitstack completion; it only checks the
      Tableau win condition.
    """
  
        
    # def check_loss(self): 


In [140]:
# program the heuristic:

# my strategy is to flip as many cards up as quick as possible
# since once all cards are face-up, the game is more or less trivially won
def nich_heuristic(cartota_move, playota_onto, board):
    """
    Compute a heuristic score for evaluating the desirability of a potential move.

    This heuristic is used to rank moves based on how strategically beneficial
    they are in the game (e.g., for an AI or hint system).

    Parameters
    ----------
    cartota_move : tuple
        A tuple describing the card being moved.
        Format: (card, (location, extra_info))
        - card: the Card object being moved.
        - location: "Tableau" or "Hand".
        - extra_info: pile index or other relevant value.

    playota_onto : tuple
        A tuple describing the card or destination being played onto.
        Format: (playon_card, (location, extra_info))
        - location: "Tableau" or "Suitstacks".

    board : Board
        The current game board object. Needed to inspect Tableau piles and
        count face-down cards.

    Returns
    -------
    int
        A heuristic score:
        - Higher values indicate more desirable moves.
        - Scores currently reflect:
            * +5 to +N for flipping a face-down card in the Tableau,
              where N = number of face-down cards (prioritizes deeper piles).
            * +5 for moves that place a card into the Suitstacks.
            * 0 for neutral moves.

    Heuristic Logic
    ---------------
    1. **Flipping face-down cards (high priority)**  
       - If playing a Tableau card that is the top face-up card of its pile  
         (position == 0), flipping a face-down card earns `5 + num_facedown`.

    2. **Moving cards to Suitstacks (moderate priority)**  
       - Any move to Suitstacks receives a score of 5.

    3. **Otherwise (tie / neutral)**  
       - Return 0.

    Notes
    -----
    - This heuristic is intentionally simple and can be extended with
      advanced scoring rules.
    """
    card = cartota_move[0]
    location = cartota_move[1][0]
    loc_extra = cartota_move[1][1]

    # play = playota_onto[0]
    play_loc = playota_onto[1][0]
    # play_extra = playota_onto[1][1]

    # if we play from the tableau
    if location == "Tableau":
        position = board.tableau.piles[loc_extra].face_up.index(card)
        num_facedown = len(board.tableau.piles[loc_extra].face_down)
        # if we can flip a face down card from the pile then we should do it
        # and we do so based on how many face down cards are in the pile;
        # we flip larger piles first
        if position == 0: return 5 + num_facedown
    
    # in general we want cards to go into suitstacks
    if play_loc == "Suitstacks": return 5

    # other scenarios are tied
    return 0
    
def nich_priority(cartota_move, playota_onto, board):#, playota_onto):
    """
    Assign a priority score to a potential move according to the Nich strategy.

    Parameters
    ----------
    cartota_move : tuple
        A tuple representing the card to move and its current location.
        Format: (card, (location_type, location_extra))
        - card : Card object being moved
        - location_type : str, one of {"Hand", "Tableau", "Suitstacks"}
        - location_extra : int or str, depending on location (pile index or suit)

    playota_onto : tuple
        A tuple representing the potential destination of the card.
        Format: (card_or_none, (destination_type, destination_extra))
        - destination_type : str, one of {"Tableau", "Suitstacks"}
        - destination_extra : int or str, depending on destination

    board : Board
        The current board state containing Hand, Tableau, and Suitstacks.

    Returns
    -------
    int
        A numeric priority value for the move:
        - Positive values indicate favorable moves.
        - Negative values indicate moves to avoid.
        - Zero indicates neutral priority.

    Behavior
    --------
    1. Cards in the Hand that are the highest value (King) are withheld (-1)
       unless a legal play exists on a braided Tableau card (+1).
    2. Moves from Hand or Suitstacks to Tableau that flip face-down cards are prioritized
       by adding the number of face-down cards (+1 + num_facedown).
    3. Cards in Suitstacks are generally kept there (-5) unless moving to Tableau flips a card.
    4. All other moves are considered neutral (0).

    Notes
    -----
    - "cartota" is a card and its current location.
    - "playota" is a potential move destination.
    - Used as the priority function in the Nich AI strategy for greedy gameplay.
    """
    card = cartota_move[0]
    location = cartota_move[1][0]
    # loc_extra = cartota_move[1][1]

    # play = playota_onto[0]
    play_loc = playota_onto[1][0]
    # play_extra = playota_onto[1][1]

    

    # we want to play our highest-ranking card last
    # and we want to withold it if nothing can be played on it
    # otherwise it's good to play
    if location == "Hand":
        if card.value == Constants().VALUES[-1]:
            available = board.check_available()
            for able in available:
                if able[0].is_braided_with(card): return 1
            return -1
        
    # IN GENERAL if we can play onto the Tableau from Hand or Suitstacks
    # then out of all of our plays we should prioritize those which flip cards
    if play_loc == "Tableau" and location != "Tableau":
        for pile in board.tableau.piles:
            if pile.is_empty(): continue
            top_card = pile.face_up[0]
            num_facedown = len(pile.face_down)
            if top_card.is_braided_with(card): return 1 + num_facedown
    
    # we generally want to keep suitstack cards there
    # unless putting it on the tableau can flip another card
    if location == "Suitstacks": return -5
    
    # any other scenarios are equally 0
    return 0

nich_strategy = (nich_heuristic,nich_priority)


In [141]:

# strategy taken from Yan et al 2005

def yan_heuristic(cartota_move, playota_onto, board):

    """
    Compute a simple heuristic score for evaluating the desirability of a move
    based on Yan's heuristic rules.

    The heuristic is intended to guide move selection by prioritizing:
    - Moving cards into the Suitstacks,
    - Playing cards from the Hand into the Tableau,
    - Avoiding pulling cards out of the Suitstacks.

    Parameters
    ----------
    cartota_move : tuple
        A tuple describing the card being played.
        Format: (card, (location, extra_info))
        - location: "Tableau", "Hand", or "Suitstacks".

    playota_onto : tuple
        A tuple describing the location or card being played onto.
        Format: (playon_card, (location, extra_info))
        - location: "Tableau" or "Suitstacks".

    board : Board
        The current game board object (not directly used in this heuristic but
        included for consistency with other heuristics).

    Returns
    -------
    int
        A heuristic score based on rules:
        - +5 for moves that place a card into the Suitstacks.
        - +5 for playing a card from the Hand into the Tableau.
        - -10 for moving a card out of the Suitstacks back to the Tableau.
        - 0 for all other moves.

    Heuristic Logic
    ---------------
    1. **play_loc == "Suitstacks" → +5**  
       Adding cards to the Suitstacks is strongly prioritized.

    2. **location == "Hand" and play_loc == "Tableau" → +5**  
       Encourages clearing the Hand onto the Tableau.

    3. **location == "Suitstacks" and play_loc == "Tableau" → -10**  
       Discourages removing cards from the Suitstacks (very undesirable).

    4. **Otherwise → 0**  
       Neutral move.

    Notes
    -----
    - This heuristic is intentionally simple and complements more detailed ones
      such as `nich_heuristic`.
    - Can be used in an AI or solver to evaluate move strength.
    """
    # card = cartota_move[0]
    location = cartota_move[1][0]
    # loc_extra = cartota_move[1][1]

    # play = playota_onto[0]
    play_loc = playota_onto[1][0]
    # play_extra = playota_onto[1][1]

    if play_loc == "Suitstacks": return 5

    elif location == "Hand" and play_loc == "Tableau": return 5

    elif location == "Suitstacks" and play_loc == "Tableau": return -10

    else: return 0
    

def yan_priority(cartota_move, playota_onto, board):
    """
    Compute a priority score for a move based on Yan's strategy rules.

    This function provides an additional layer of scoring beyond the basic
    heuristic, allowing the AI to prefer moves that flip face-down cards or
    correctly place cards from the Hand.

    Parameters
    ----------
    cartota_move : tuple
        A tuple describing the card being moved.
        Format: (card, (location, extra_info))
        - card: Card object.
        - location: "Tableau", "Hand", or "Suitstacks".
        - extra_info: pile index if coming from Tableau.

    playota_onto : tuple
        A tuple describing the target.
        Format: (playon_card, (location, extra_info))
        - location: "Tableau" or "Suitstacks".

    board : Board
        The current game board, required for checking tableau piles and available moves.

    Returns
    -------
    int
        A priority score:
        - Larger numbers = higher priority.
        - Score meanings:
            * +N (N > 1): Flipping a face-down card (higher if the face-down pile is deeper).
            * +1: Good Hand → Tableau play.
            * -1: Discourages playing the highest-value card (e.g., King) from Hand when it cannot braid.
            * 0: Neutral moves.

    Priority Logic
    --------------
    1. **Playing onto Tableau from Tableau**
       - If the card is the top face-up card (position == 0) AND
         the pile has face-down cards below it,
         reward = (num_facedown + 1).

    2. **Playing from Hand to Tableau**
       - If the card is the highest-ranked card (e.g., King),
         check if ANY tableau pile can braid with it:
            - If yes → priority = 1
            - If no → priority = -1
       - For all other cards from the hand → priority = 1

    3. **All other situations**
       - Priority = 0

    Notes
    -----
    - This priority function is intended to complement `yan_heuristic`.
    - The final strategy is a tuple stored as `yan_strategy`, combining both
      heuristic and priority functions.
    """

    card = cartota_move[0]
    location = cartota_move[1][0]
    loc_extra = cartota_move[1][1]

    # play = playota_onto[0]
    play_loc = playota_onto[1][0]
    # play_extra = playota_onto[1][1]

    if play_loc == "Tableau":

        if location == "Tableau":
            position = board.tableau.piles[loc_extra].face_up.index(card)
            num_facedown = len(board.tableau.piles[loc_extra].face_down)
            if position == 0 and num_facedown > 0: return len(board.tableau.piles[loc_extra].face_down)+1
        
        elif location == "Hand":
            if card.value == Constants().VALUES[-1]:
                for able in board.check_available():
                    if able[0].is_braided_with(card): return 1
                
                return -1

            else: return 1
    
    return 0

yan_strategy = (yan_heuristic,yan_priority)

